In [ ]:
!pip install "gspread==6.1.4"

In [ ]:
sheets_link = "https://docs.google.com/spreadsheets/d/1oLIFXvFUiqP38fEqpEz1NyxMJ7RHHmdOGxMW2vEcZRo/edit?gid=0#gid=0"

In [ ]:
import gspread
from google.oauth2.service_account import Credentials

scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

credentials = Credentials.from_service_account_file('analysis-ml-project.json', scopes=scope)
gc = gspread.authorize(credentials)

sh = gc.open_by_key("1tyxACc95GD88T2Me_xhktYbc14P6-BBZkOWlT7MUaeU")

In [ ]:
import pandas as pd

worksheet = sh.sheet1

data = worksheet.get_all_values()

df = pd.DataFrame(data[1:], columns=data[0])

df.head()

,id,subject,body,answer,type,queue,priority,language,business_type,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,tag_9
0,1001352387736,Urgent: Critical impact on enterprise network ...,"Dear Customer Support Team, We are experiencin...",Subject: Re: Urgent: Critical impact on enterp...,Incident,Technical Support,high,en,IT Services,Service Outage,Network Issue,Urgent Issue,Technical Support,Problem Resolution,Critical Failure,System Crash,Service Disruption,
1,1004699418379,Intermittent Cursor Freezing Issue on Dell XPS,"Dear Customer Support,<br><br>I hope this mess...","Dear <name>,\n\n\nThank you for reaching out r...",Incident,Product Support,low,en,Tech Online Store,Technical Support,Product Support,Hardware Failure,Problem Resolution,Urgent Issue,Service Recovery,Documentation Request,,
2,1006966905046,Dringend: Unterstützung für die Datenwiederher...,"Hallo, wir haben severe Datenverluste in MySQL...","Hallo, vielen Dank, dass Sie uns kontaktiert h...",Incident,Technical Support,high,de,IT Services,Data Breach,Backup Restore,Technical Support,Urgent Issue,Software Bug,Problem Resolution,,,
3,1009231330404,Anfrage zu den MacBook Air M1 Funktionen,"Sehr geehrtes Kundenserviceteam,\n\n\nich hoff...","Sehr geehrter <name>,\n\n\nvielen Dank für Ihr...",Request,Sales and Pre-Sales,low,de,Tech Online Store,Customer Service,Product Support,Sales Inquiry,Technical Guidance,Warranty Claim,General Inquiry,,,
4,1024440081041,Solicitação de Assistência com Erro de Instala...,"Caro Suporte ao Cliente,\n\n\nEstou enfrentand...","Caro <name>,\n\n\nObrigado por entrar em conta...",Problem,Technical Support,medium,pt,IT Services,Technical Support,Software Bug,Urgent Issue,Problem Resolution,Product Support,,,,


In [ ]:
!pip install langdetect

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from langdetect import detect
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Encoding categorical columns (language, queue, issue_type)
def encode_categorical_columns(df):
    le_lang = LabelEncoder()
    le_queue = LabelEncoder()
    le_type = LabelEncoder()

    df['lang_encoded'] = le_lang.fit_transform(df['language'])
    df['queue_encoded'] = le_queue.fit_transform(df['queue'])
    df['type_encoded'] = le_type.fit_transform(df['type'])

    return df, le_lang, le_queue, le_type

# Perform clustering using encoded language, queue, and issue type
def perform_clustering(df, n_clusters):
    features = df[['lang_encoded', 'queue_encoded', 'type_encoded']]
    kmeans = KMeans(n_clusters=n_clusters)
    df['cluster'] = kmeans.fit_predict(features)
    return df, kmeans

# Generate a response template based on the cluster and escalation logic
def generate_response(issue_description, df, kmeans, le_lang, le_queue, le_type):
    try:
        lang_pred = detect(issue_description)
    except:
        lang_pred = 'en'

    queue_pred = 'Technical Support'
    type_pred = 'Incident'

    lang_pred_encoded = le_lang.transform([lang_pred])[0]
    queue_pred_encoded = le_queue.transform([queue_pred])[0]
    type_pred_encoded = le_type.transform([type_pred])[0]

    # Create a DataFrame for prediction with correct feature names
    prediction_data = pd.DataFrame([[lang_pred_encoded, queue_pred_encoded, type_pred_encoded]],
                                   columns=['lang_encoded', 'queue_encoded', 'type_encoded'])

    # Predict the cluster using the DataFrame with feature names
    cluster_pred = kmeans.predict(prediction_data)[0]

    # Escalation triggers
    escalation_keywords = ["urgent", "critical", "network down", "system outage"]
    escalation_clusters = [2]
    similarity_threshold = 0.7

    # Escalation logic:
    if any(keyword in issue_description.lower() for keyword in escalation_keywords) or \
            cluster_pred in escalation_clusters:
        escalation_record = {
            "issue_description": issue_description,
            "predicted_language": lang_pred,
            "predicted_queue": queue_pred,
            "predicted_type": type_pred,
            "cluster": cluster_pred
        }
        print("Issue Escalated:", escalation_record)
        return "Thank you for contacting us. This issue has been escalated and will be addressed by a specialist shortly."

    # Filter the data by predicted cluster
    cluster_data = df[df['cluster'] == cluster_pred]

    # If no data for the cluster, return a default response and escalate
    if cluster_data.empty:
        print("Issue Escalated: No data for predicted cluster")
        return "Thank you for contacting us. We are reviewing your request and will get back to you soon."

    cluster_data_lang_filtered = cluster_data[cluster_data['language'] == lang_pred]

    # If no data for the language within the cluster, return a default response and escalate
    if cluster_data_lang_filtered.empty:
        print("Issue Escalated: No data for predicted language within cluster")
        return "Thank you for contacting us. We are reviewing your request and will get back to you soon."

    # Use TF-IDF and cosine similarity to find the most similar answer
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(cluster_data_lang_filtered['answer'].tolist() + [issue_description])
    similarity_scores = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1]).flatten()

    # Similarity-based escalation:
    if max(similarity_scores) < similarity_threshold:
        escalation_record = {
            "issue_description": issue_description,
            "predicted_language": lang_pred,
            "predicted_queue": queue_pred,
            "predicted_type": type_pred,
            "cluster": cluster_pred,
            "max_similarity": max(similarity_scores)
        }
        print("Issue Escalated (Low Similarity):", escalation_record)
        return "Thank you for contacting us. We are reviewing your request and will get back to you soon."

    # If no escalation is triggered, return the most similar answer
    most_similar_idx = similarity_scores.argmax()
    response = cluster_data_lang_filtered['answer'].iloc[most_similar_idx]
    return f"Thank you for contacting us. Based on your issue, here's the most relevant response: \n\n{response}"


# Main script that processes the data
def main():
    # Assuming 'data1' is your DataFrame
    # Encode the categorical columns
    encoded_data1, le_lang, le_queue, le_type = encode_categorical_columns(data1)

    # Perform clustering based on the encoded features
    n_clusters = 3
    clustered_data1, kmeans = perform_clustering(encoded_data1, n_clusters)

    # Test with a sample issue
    test_issue = "Our network is down, and we need urgent assistance!"
    # test_issue = "I need help resetting my password."

    # Generate a response for the sample issue
    response = generate_response(test_issue, clustered_data1, kmeans, le_lang, le_queue, le_type)

    # Output the response
    print("Automated Response:", response)

# Run the main function
if __name__ == "__main__":
    main()

Issue Escalated: {'issue_description': 'Our network is down, and we need urgent assistance!', 'predicted_language': 'en', 'predicted_queue': 'Technical Support', 'predicted_type': 'Incident', 'cluster': 1}
Automated Response: Thank you for contacting us. This issue has been escalated and will be addressed by a specialist shortly.


In [ ]:
df_en = df[df['language'] == 'en']
all_tags = []
for index, row in df_en.iterrows():
    tags = []
    for i in range(1, 4):  # Iterate through tag_1 to tag_3
        tag_value = row[f"tag_{i}"]
        if pd.notna(tag_value) and tag_value != '':
            tags.extend(str(tag_value).split())
    all_tags.extend(tags)

# Calculate value counts for the combined tags
from collections import Counter
tag_counts = Counter(all_tags)
print(tag_counts.most_common(50))

[('Support', 447), ('Technical', 262), ('Issue', 215), ('Urgent', 133), ('Service', 105), ('Product', 104), ('IT', 98), ('Network', 51), ('Disruption', 48), ('Software', 47), ('Bug', 47), ('Failure', 35), ('Hardware', 34), ('Customer', 29), ('Problem', 26), ('Resolution', 26), ('Billing', 24), ('Inquiry', 24), ('Outage', 20), ('General', 16), ('Account', 16), ('Assistance', 16), ('Performance', 15), ('Tuning', 15), ('Guidance', 15), ('System', 14), ('Returns', 11), ('and', 11), ('Exchanges', 11), ('Request', 11), ('Maintenance', 10), ('Incident', 9), ('Report', 9), ('Sales', 8), ('Payment', 6), ('Processing', 6), ('Feature', 6), ('Notification', 5), ('Warranty', 5), ('Claim', 5), ('Crash', 5), ('Setup', 4), ('Refund', 4), ('Order', 4), ('Printer', 3), ('Login', 3), ('Replacement', 2), ('Wireless', 2), ('Services', 2), ('Cost', 2)]


In [ ]:
import pandas as pd
from collections import Counter

def should_escalate(incoming_issue):
    if incoming_issue["priority"] == "high":
        tags_combined = " ".join([incoming_issue[f"tag_{i+1}"] for i in range(0, 3)])
        for key in ["Disruption", "Failure", "Outage", "Incident", "Crash", "Breach","Urgent", "Critical"]:
            if key.lower() in tags_combined.lower():
                return True
    return False

# Sample
incoming_issue_data = {
    "priority": "high",
    "tag_1": "Network",
    "tag_2": "Outage",
    "tag_3": "Critical"
}
#if escalation is needed or not
escalation_needed = should_escalate(incoming_issue_data)
print(f"Escalation needed: {escalation_needed}")


Escalation needed: True


In [ ]:
def escalate_all_rows(data):
    """Escalates issues for all rows in the DataFrame based on defined criteria."""

    escalated_rows = []
    for index, row in df.iterrows():
        issue = row.to_dict()
        if should_escalate(issue):
            issue["escalated"] = True
            issue = adjust_priority_based_on_severity(issue)
            issue["escalation_score"] = escalation_score(issue)
            escalated_rows.append(issue)
        else:
            issue["escalated"] = False
            issue["escalation_score"] = escalation_score(issue)
            escalated_rows.append(issue)

    return pd.DataFrame(escalated_rows)

# Assuming 'df_en' is your DataFrame (from the previous code)
df_escalated = escalate_all_rows(df_en)

# Convert the DataFrame back to a list of lists for writing to Google Sheets
updated_data = df_escalated.values.tolist()


In [ ]:
# prompt: insert headers and update the google sheet with the updated data

# Insert headers and update the Google Sheet
headers = df_escalated.columns.tolist()
updated_data = [headers] + updated_data  # Prepend the headers to the data

# Update the worksheet with the new data
worksheet.update('A1', updated_data)

<ipython-input-13-3963f2f492b1>:8: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update('A1', updated_data)


APIError: APIError: [403]: The caller does not have permission

In [ ]:
# prompt: Escalate Issue (/check_escalate) api calling function to be used.
import requests
import json

def escalate_issue(issue_data, api_endpoint):
    """
    Sends a POST request to the /check_escalate API endpoint to escalate an issue.

    Args:
        issue_data (dict): A dictionary containing the issue data.
        api_endpoint (str): The URL of the /check_escalate API endpoint.

    Returns:
        dict: The JSON response from the API, or None if there was an error.
    """
    try:
        headers = {'Content-type': 'application/json'}  # Set the content type to JSON
        response = requests.post(api_endpoint, data=json.dumps(issue_data), headers=headers)
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

        return response.json()

    except requests.exceptions.RequestException as e:
        print(f"Error escalating issue: {e}")
        return None

# Example Usage (replace with your actual API endpoint and issue data)
api_url = "your_api_endpoint/check_escalate"

# Assuming 'df_escalated' is your DataFrame with the 'escalated' column (from the previous code)
for index, row in df_escalated.iterrows():
    if row['escalated']:
        issue_data = row.to_dict()  # Convert the row to a dictionary
        escalation_response = escalate_issue(issue_data, api_url)

        if escalation_response:
            print(f"Escalation successful for issue {index}: {escalation_response}")
        else:
            print(f"Escalation failed for issue {index}")


Error escalating issue: Invalid URL 'your_api_endpoint/check_escalate': No scheme supplied. Perhaps you meant https://your_api_endpoint/check_escalate?
Escalation failed for issue 0
Error escalating issue: Invalid URL 'your_api_endpoint/check_escalate': No scheme supplied. Perhaps you meant https://your_api_endpoint/check_escalate?
Escalation failed for issue 2
Error escalating issue: Invalid URL 'your_api_endpoint/check_escalate': No scheme supplied. Perhaps you meant https://your_api_endpoint/check_escalate?
Escalation failed for issue 7
Error escalating issue: Invalid URL 'your_api_endpoint/check_escalate': No scheme supplied. Perhaps you meant https://your_api_endpoint/check_escalate?
Escalation failed for issue 8
Error escalating issue: Invalid URL 'your_api_endpoint/check_escalate': No scheme supplied. Perhaps you meant https://your_api_endpoint/check_escalate?
Escalation failed for issue 10
Error escalating issue: Invalid URL 'your_api_endpoint/check_escalate': No scheme supplie